In [18]:
%load_ext aiida
%aiida

The aiida extension is already loaded. To reload it, use:
  %reload_ext aiida


Loaded AiiDA DB environment - profile name: bit.

In [19]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from matchest.aiida_utils.pmg import load_mp_struct
from ase.symbols import Formula
from aiida.engine import submit

from matchest.aiida_utils.workflows.simple_vac import SimpleVacancyWorkChain

from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pymatgen.core import Composition

In [20]:
basepath = GroupPathX('hc-defect')
workpath = basepath['workflows']
elemental_struct_path = GroupPathX('defects/elemental_ref')

In [21]:
results = defaultdict(lambda : {})
for path in workpath:
    node = path.node
    if node.is_finished_ok:
        for kind, eng in node.outputs.vacancy_formation_energies.items():
            ref_structure = node.inputs.relax.structure.get_ase()
            form = ref_structure.symbols.formula.reduce()[0]
            #print(ref_structure.symbols.formula.reduce()[0], kind, eng[kind])
            results[str(form)][kind] = eng[kind]

Find the minimum as maxium V formation energy for cations

In [22]:
anions = ['Sb', 'Bi', 'As', 'P', 'Se', 'O']

In [23]:
min_max = []
for key,value in results.items():
    
    symbols = list(set(Formula(key)))
    comp = Composition(key)
    sorted_elems = sorted(comp.elements, key=lambda x : x.electron_affinity)
    if len(sorted_elems) == 2:
        # Discard the element with the higest electron affinity
        no_anion = [x.symbol for x in sorted_elems[:-1]]
    else:
        no_anion = [elem for elem in sorted_elems if elem.symbol not in anions]
    #print(f"{key}  max: {max(engs):.3f} eV min: {min(engs):.3f} eV")
    #print(sorted_elems)
    names, engs = zip(*[[f'V_{a}' , value[f'V_{a}']] for a in no_anion])
    min_max.append([key, max(engs), min(engs), names[np.argmax(engs)], names[np.argmin(engs)]])
    
df = pd.DataFrame(min_max, columns=['name', 'E_form_max', 'E_form_min', 'E_form_max_element', 'E_form_min_element'])
df = df.set_index('name')
df

,E_form_max,E_form_min,E_form_max_element,E_form_min_element
name,,,,
PbZnSb2,1.770830,1.087918,V_Pb,V_Zn
SnCdSb2,2.028047,1.241505,V_Sn,V_Cd
GeCdSb2,1.258318,1.115496,V_Ge,V_Cd
SnZnBi2,1.231478,0.278016,V_Sn,V_Zn
GeZnBi2,0.632074,0.227689,V_Ge,V_Zn
GeCdAs2,2.363226,1.837619,V_Ge,V_Cd
GeZnSb2,1.489911,0.971533,V_Ge,V_Zn
PbCdAs2,1.711863,1.693158,V_Cd,V_Pb
PbCdP2,2.519697,0.884824,V_Cd,V_Pb


## Print the formation energies for easy copying into the summary feishu sheet


In [24]:
cases="""InAuSe2
TlCuSe2
PbZnSb2
PbZnSb2
GeCdSb2
SnCdSb2
PbCdP2
PbCdAs2
PbZnAs2
GeCdSb2
Mg3PbO
Ba3SnO
Ba3PbO
Mg3Bi2
Mg3Bi2
Sr2Pb
Ca2Pb"""
for value in df.loc[[x for x in cases.split('\n')]].E_form_min:
    print(f'{value:.3f}')

-0.284
0.581
1.088
1.088
1.115
1.242
0.885
1.693
1.800
1.115
0.225
1.640
1.538
1.166
1.166
2.045
1.604


In [17]:
cases="""InCuSe2
InCuSe2
SnZnSb2
GeZnSb2
GeZnSb2
SnZnSb2
PbZnP2
GeCdAs2
PbZnP2
GeCdAs2
Ca3PbO
Sr3SnO
Sr3PbO
Ca3Bi2
Sr3Bi2
Ca2Pb
Ca2Sn"""
for value in df.loc[[x for x in cases.split('\n')]].E_form_min:
    print(f'{value:.3f}')

0.419
0.419
1.101
0.972
0.972
1.101
2.657
1.838
2.657
1.838
1.893
1.745
1.557
2.673
2.823
1.604
1.843
